In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account

#### Create client and link Google BigQuery dataset.

In [ ]:
def create_biguery_client ():
    # Set up authentication using a service account
    credentials = service_account.Credentials.from_service_account_file('bq_key.json')
    # Create a BigQuery client using the credentials
    client = bigquery.Client(credentials=credentials, project=credentials.project_id)
    print(f"The Client:\n ${client}\n was successfully created.")
    return client


In [ ]:
client = create_biguery_client()

#### Construct a reference to a BigQuery dataset.

stackoverflow

In [ ]:
project_id = "bigquery-public-data"
dataset_id = "stackoverflow" 
def create_dataset(client):
    # Construct a reference to the "stackoverflow" dataset
    dataset_ref = client.dataset(f"{dataset_id}", project=f"{project_id}")

    # API request - fetch the dataset
    dataset = client.get_dataset(dataset_ref)
    print(f"The dataset:\n ${dataset}\n has been successfully created ")
    return dataset_ref, dataset

In [ ]:
dataset, dataset_ref = create_dataset(client)

#### Explore the data.

List the available tables.

In [ ]:
# Get a list of available tables
tables = list(client.list_tables(dataset))
list_of_tables = [table.table_id for table in tables]

# Print the list of tables (uncomment print type for debug)
# print(list_of_tables)
# print(type(list_of_tables))
for table in list_of_tables:
    print(table)


Query for how long it took in seconds to answer questions in 2018.


Dry run to validate query and test number of bytes.

In [ ]:
first_query = """
SELECT q.id AS q_id,
    MIN(TIMESTAMP_DIFF(a.creation_date, q.creation_date, SECOND)) as time_to_answer
FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
    LEFT JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
    ON q.id = a.parent_id
WHERE q.creation_date >= '2018-01-01' AND q.creation_date < '2018-02-01'
GROUP BY q.id
ORDER BY time_to_answer
    """
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(first_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

Run the query with safe_config.

In [ ]:
# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(first_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
first_results = my_query_job.to_dataframe()
# print(results)
print("Percentage of answered questions: %s%%" % \
    (sum(first_results["time_to_answer"].notnull()) / len(first_results) * 100))
print("Number of questions:", len(first_results))
first_results.head()


In [ ]:
first_results.describe()
first_results["time_to_answer"].describe()

Query to see what users posts questions or answered questions in January 2019 with JOIN.

Dry run to validate query and test byte size.

In [ ]:
three_tables_query = """
 SELECT u.id AS id,
     MIN(q.creation_date) AS q_creation_date,
     MIN(a.creation_date) AS a_creation_date
 FROM `bigquery-public-data.stackoverflow.users` AS u
     FULL JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
         ON u.id = a.owner_user_id
     LEFT JOIN `bigquery-public-data.stackoverflow.posts_questions` AS q
         ON q.owner_user_id = u.id
 WHERE u.creation_date >= '2019-01-01' and u.creation_date < '2019-02-01'
 GROUP BY id
    """
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(first_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

Run the query with safe_config.

In [ ]:
# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(three_tables_query, job_config=safe_config)
    
# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()
# print(results)
results.head()

Query distinct users who posted on January 1, 2019 with UNION.

Dry run to validate query and test byte size.

In [ ]:
project_id = "bigquery-public-data"
dataset_id = "stackoverflow"    
all_users_query = f"""
    SELECT q.owner_user_id, 
    FROM `{project_id}.{dataset_id}.posts_questions` AS q
    WHERE EXTRACT(DATE FROM q.creation_date) = '2019-01-01'
    UNION DISTINCT
    SELECT a.owner_user_id
    FROM `{project_id}.{dataset_id}.posts_answers` AS a
    WHERE EXTRACT(DATE FROM a.creation_date) = '2019-01-01'
"""
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(all_users_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

Run the query with safe_config.

In [ ]:
# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(all_users_query, job_config=safe_config)
    
# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()
# print(results)
results.head()